# Heat map from an open Excel workbook

This notebook reads directly from a workbook that is **already open** in desktop Excel. Select a workbook and sheet, then choose an Excel table or enter a sheet range with a header row. Choose the source columns for X, Y, and (optionally) weights. The plot bins X/Y observations and colors each bin by count, sum of weights, or average weight.

### Requirements
- Windows with desktop Microsoft Excel running and the source workbook open.
- Run this notebook in a Python environment with `xlwings`, `ipywidgets`, `pandas`, `numpy`, and `matplotlib` installed. Install missing packages in that environment with `python -m pip install xlwings ipywidgets pandas numpy matplotlib`.
- In VS Code, use a Jupyter-enabled Python kernel. Excel automation uses xlwings/COM; this notebook does not work with Excel for the web or a closed workbook.

For an A1 source range, include one header row (for example `A1:C500`). X and Y ranges can be selected as columns below. Bin definitions accept either a positive integer count (such as `20`) or explicit comma-separated numeric edges (such as `0, 5, 10, 20`). The colormap selector lists every colormap registered by Matplotlib; its gradient preview updates with the selected map and reverse option.

In [ ]:
import math
import copy
import io
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xlwings as xw
import ipywidgets as widgets
from IPython.display import clear_output, display


def open_workbook_entries():
    """Return display labels mapped to Excel process IDs and workbook names."""
    entries = {}
    for app in xw.apps:
        for book in app.books:
            label = f"{book.name} (Excel PID {app.pid})"
            entries[label] = (app.pid, book.name)
    return entries


def selected_book():
    if workbook_select.value not in workbook_entries:
        raise ValueError("Select an open workbook, then refresh the workbook list if needed.")
    pid, name = workbook_entries[workbook_select.value]
    return xw.apps[pid].books[name]


def refresh_sheets():
    book = selected_book()
    sheet_select.options = [sheet.name for sheet in book.sheets]
    if sheet_select.options:
        sheet_select.value = sheet_select.options[0]
    refresh_tables()


def refresh_tables():
    if not sheet_select.options:
        table_select.options = []
        return
    sheet = selected_book().sheets[sheet_select.value]
    table_select.options = [table.name for table in sheet.tables]


def refresh_workbooks(_=None):
    global workbook_entries
    workbook_entries = open_workbook_entries()
    workbook_select.options = list(workbook_entries)
    if workbook_select.options:
        workbook_select.value = workbook_select.options[0]
        refresh_sheets()
        status.value = "<span style='color:green'>Open workbooks found. Select a source and load its columns.</span>"
    else:
        sheet_select.options = []
        table_select.options = []
        status.value = "<span style='color:#a60'>No open Excel workbooks found. Open a workbook in desktop Excel, then refresh.</span>"


def on_workbook_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        refresh_sheets()


def on_sheet_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        refresh_tables()


def on_table_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        table = selected_book().sheets[sheet_select.value].tables[change['new']]
        source_range.value = table.range.address
        status.value = "Table range selected. Load columns to refresh the X/Y/weight choices."


def read_source_frame(sheet, address):
    if table_select.value:
        table = sheet.tables[table_select.value]
        if address == table.range.address:
            header_range = table.header_row_range
            if header_range is None:
                raise ValueError("The selected Excel table does not have a visible header row.")
            headers = header_range.options(ndim=2).value[0]
            body_range = table.data_body_range
            rows = [] if body_range is None else body_range.options(ndim=2).value
            return pd.DataFrame(rows, columns=headers)
    return sheet.range(address).options(pd.DataFrame, header=1, index=False).value


def load_columns(_=None):
    if not sheet_select.options:
        raise ValueError("Select an open workbook and sheet first.")
    address = source_range.value.strip()
    if not address:
        raise ValueError("Choose a table or enter an A1 source range with a header row.")
    sheet = selected_book().sheets[sheet_select.value]
    frame = read_source_frame(sheet, address)
    if frame is None or frame.empty or len(frame.columns) == 0:
        raise ValueError("The selected source contains no data columns. Check the range and header row.")
    if not frame.columns.is_unique:
        raise ValueError("The source has duplicate column headers. Rename them in Excel before loading.")
    columns = [str(column) for column in frame.columns]
    column_pairs = [(str(column), column) for column in frame.columns]
    x_select.options = column_pairs
    y_select.options = column_pairs
    weight_select.options = [("(none)", "")] + column_pairs
    if len(columns) >= 2:
        x_select.value = frame.columns[0]
        y_select.value = frame.columns[1]
    if len(columns) >= 3:
        weight_select.value = frame.columns[2]
    else:
        weight_select.value = ""
    status.value = f"Loaded {len(frame)} data rows and {len(columns)} columns."


def parse_bins(specification, values, axis_label):
    text = specification.strip()
    if not text:
        raise ValueError(f"Enter a bin count or explicit edges for {axis_label}.")
    if re.fullmatch(r"\d+", text):
        count = int(text)
        if count < 1:
            raise ValueError(f"{axis_label} bin count must be at least 1.")
        return np.histogram_bin_edges(values, bins=count)
    try:
        edges = np.asarray([float(part.strip()) for part in text.split(",")], dtype=float)
    except ValueError as exc:
        raise ValueError(f"{axis_label} bins must be an integer count or comma-separated numeric edges.") from exc
    if len(edges) < 2 or not np.isfinite(edges).all() or not np.all(np.diff(edges) > 0):
        raise ValueError(f"{axis_label} bin edges must contain at least two finite, strictly increasing values.")
    return edges


def parse_optional_limit(text, label):
    text = text.strip()
    if not text:
        return None
    try:
        value = float(text)
    except ValueError as exc:
        raise ValueError(f"{label} must be a finite number or left blank.") from exc
    if not np.isfinite(value):
        raise ValueError(f"{label} must be a finite number or left blank.")
    return value


def validate_limits(lower, upper, label):
    if lower is not None and upper is not None and lower >= upper:
        raise ValueError(f"{label} minimum must be less than its maximum.")


def update_cmap_preview(_=None):
    cmap = copy.copy(plt.get_cmap(cmap_select.value))
    if invert_cmap.value:
        cmap = cmap.reversed()
    gradient = np.linspace(0, 1, 512).reshape(1, -1)
    fig, ax = plt.subplots(figsize=(4, 0.3))
    ax.imshow(gradient, aspect="auto", cmap=cmap)
    ax.set_axis_off()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    cmap_preview.value = buffer.getvalue()


def plot_heatmap(_=None):
    plot_output.clear_output(wait=True)
    try:
        if not sheet_select.options:
            raise ValueError("Select an open workbook and sheet first.")
        address = source_range.value.strip()
        if not address:
            raise ValueError("Choose a table or enter an A1 source range, then load its columns.")
        if x_select.value is None or y_select.value is None:
            raise ValueError("Load the source columns and select both X and Y columns.")
        sheet = selected_book().sheets[sheet_select.value]
        frame = read_source_frame(sheet, address)
        if frame is None or frame.empty:
            raise ValueError("The selected source contains no data rows.")
        x = pd.to_numeric(frame[x_select.value], errors="coerce").to_numpy(dtype=float)
        y = pd.to_numeric(frame[y_select.value], errors="coerce").to_numpy(dtype=float)
        aggregation = aggregate_select.value
        use_weights = aggregation != "Bin count" and weight_select.value != ""
        weights = None
        valid = np.isfinite(x) & np.isfinite(y)
        if use_weights:
            weights = pd.to_numeric(frame[weight_select.value], errors="coerce").to_numpy(dtype=float)
            valid &= np.isfinite(weights)
        x, y = x[valid], y[valid]
        if weights is not None:
            weights = weights[valid]
        if x.size == 0:
            raise ValueError("No rows have usable numeric X/Y values (and numeric weights, when selected).")
        x_edges = parse_bins(x_bins.value, x, "X")
        y_edges = parse_bins(y_bins.value, y, "Y")
        if aggregation == "Bin count":
            values, _, _ = np.histogram2d(x, y, bins=(x_edges, y_edges))
            color_label = "Bin count"
        else:
            if weights is None:
                raise ValueError("Select a weight column for sum or average aggregation.")
            sums, _, _ = np.histogram2d(x, y, bins=(x_edges, y_edges), weights=weights)
            if aggregation == "Sum of weights":
                values = sums
                color_label = "Sum of weights"
            else:
                counts, _, _ = np.histogram2d(x, y, bins=(x_edges, y_edges))
                values = np.divide(sums, counts, out=np.full_like(sums, np.nan), where=counts > 0)
                color_label = "Average of weights"
        x_min = parse_optional_limit(x_min_input.value, "X-axis minimum")
        x_max = parse_optional_limit(x_max_input.value, "X-axis maximum")
        y_min = parse_optional_limit(y_min_input.value, "Y-axis minimum")
        y_max = parse_optional_limit(y_max_input.value, "Y-axis maximum")
        legend_min = parse_optional_limit(legend_min_input.value, "Legend minimum")
        legend_max = parse_optional_limit(legend_max_input.value, "Legend maximum")
        validate_limits(x_min, x_max, "X-axis")
        validate_limits(y_min, y_max, "Y-axis")
        validate_limits(legend_min, legend_max, "Legend")
        with plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 6))
            cmap = copy.copy(plt.get_cmap(cmap_select.value))
            if invert_cmap.value:
                cmap = cmap.reversed()
            cmap.set_under(under_color.value)
            cmap.set_over(over_color.value)
            image = ax.pcolormesh(
                x_edges,
                y_edges,
                np.ma.masked_invalid(values.T),
                shading="auto",
                cmap=cmap,
                vmin=legend_min,
                vmax=legend_max,
            )
            extend = (
                "both" if legend_min is not None and legend_max is not None
                else "min" if legend_min is not None
                else "max" if legend_max is not None
                else "neither"
            )
            fig.colorbar(image, ax=ax, label=color_label, extend=extend)
            ax.set_xlabel(str(x_select.value))
            ax.set_ylabel(str(y_select.value))
            ax.set_title(f"{color_label} heat map")
            ax.set_xlim(left=x_min, right=x_max)
            ax.set_ylim(bottom=y_min, top=y_max)
            fig.tight_layout()
            plt.show()
        status.value = f"Plotted {x.size} valid observations across {len(x_edges) - 1} × {len(y_edges) - 1} bins."
    except (ValueError, TypeError, KeyError, IndexError) as exc:
        status.value = f"<span style='color:#b00'>{exc}</span>"


workbook_entries = {}
workbook_select = widgets.Dropdown(description="Workbook:", options=[], layout=widgets.Layout(width="700px"))
refresh_button = widgets.Button(description="Refresh workbooks", icon="refresh")
sheet_select = widgets.Dropdown(description="Sheet:", options=[], layout=widgets.Layout(width="500px"))
table_select = widgets.Dropdown(description="Excel table:", options=[], layout=widgets.Layout(width="500px"))
source_range = widgets.Text(description="Source range:", placeholder="e.g. A1:C500", layout=widgets.Layout(width="500px"))
load_button = widgets.Button(description="Load columns", button_style="info")
x_select = widgets.Dropdown(description="X column:", options=[], layout=widgets.Layout(width="500px"))
y_select = widgets.Dropdown(description="Y column:", options=[], layout=widgets.Layout(width="500px"))
weight_select = widgets.Dropdown(description="Weight:", options=[("(none)", "")], layout=widgets.Layout(width="500px"))
x_bins = widgets.Text(value="20", description="X bins:", layout=widgets.Layout(width="500px"))
y_bins = widgets.Text(value="20", description="Y bins:", layout=widgets.Layout(width="500px"))
aggregate_select = widgets.Dropdown(options=["Bin count", "Sum of weights", "Average of weights"], value="Bin count", description="Aggregate:", layout=widgets.Layout(width="500px"))
cmap_select = widgets.Dropdown(
    options=sorted(plt.colormaps()),
    value="viridis",
    description="Colormap:",
    layout=widgets.Layout(width="500px"),
)
invert_cmap = widgets.Checkbox(value=False, description="Reverse gradient")
cmap_preview = widgets.Image(format="png", layout=widgets.Layout(width="400px", height="30px"))
x_min_input = widgets.Text(description="X min:", placeholder="auto", layout=widgets.Layout(width="250px"))
x_max_input = widgets.Text(description="X max:", placeholder="auto", layout=widgets.Layout(width="250px"))
y_min_input = widgets.Text(description="Y min:", placeholder="auto", layout=widgets.Layout(width="250px"))
y_max_input = widgets.Text(description="Y max:", placeholder="auto", layout=widgets.Layout(width="250px"))
legend_min_input = widgets.Text(description="Legend min:", placeholder="auto", layout=widgets.Layout(width="250px"))
legend_max_input = widgets.Text(description="Legend max:", placeholder="auto", layout=widgets.Layout(width="250px"))
under_color = widgets.ColorPicker(description="Below range:", value="#440154", concise=False, layout=widgets.Layout(width="250px"))
over_color = widgets.ColorPicker(description="Above range:", value="#fde725", concise=False, layout=widgets.Layout(width="250px"))
plot_button = widgets.Button(description="Plot heat map", button_style="success", icon="line-chart")
status = widgets.HTML()
plot_output = widgets.Output()

workbook_select.observe(on_workbook_change, names="value")
sheet_select.observe(on_sheet_change, names="value")
table_select.observe(on_table_change, names="value")
cmap_select.observe(update_cmap_preview, names="value")
invert_cmap.observe(update_cmap_preview, names="value")
refresh_button.on_click(refresh_workbooks)
load_button.on_click(load_columns)
plot_button.on_click(plot_heatmap)
update_cmap_preview()

display(widgets.VBox([
    widgets.HTML("<b>1. Choose an open workbook and sheet</b>"),
    widgets.HBox([workbook_select, refresh_button]), sheet_select,
    widgets.HTML("<b>2. Select a table or enter a source range (include headers)</b>"),
    table_select, source_range, load_button,
    widgets.HTML("<b>3. Choose data columns and heat-map settings</b>"),
    x_select, y_select, weight_select,
    x_bins, y_bins, aggregate_select,
    widgets.VBox([widgets.HBox([cmap_select, invert_cmap]), cmap_preview]),
    widgets.HTML("<b>Axis ranges (leave blank for automatic limits)</b>"),
    widgets.HBox([x_min_input, x_max_input]),
    widgets.HBox([y_min_input, y_max_input]),
    widgets.HTML("<b>Legend range and out-of-range colors</b>"),
    widgets.HBox([legend_min_input, legend_max_input]),
    widgets.HBox([under_color, over_color]),
    plot_button,
    status, plot_output
]))
refresh_workbooks()

## Notes on the plot

- Empty bins are shown as blank. Values outside explicitly entered bin edges are not included.
- Rows with non-numeric or missing X/Y values are ignored; for weighted aggregations, rows with non-numeric or missing weights are also ignored.
- The average is the arithmetic mean of the selected weight values in each bin. Bin count counts observations, regardless of whether a weight column is selected.
- X/Y axis limits and legend limits are independent; leave a limit blank for automatic scaling. A legend minimum/maximum colors values below/above those bounds with the selected under/over colors.